# 03 — Modélisation

## Objectif
Entraîner un modèle de classification pour prédire la probabilité de défaut de remboursement (`TARGET = 1`).

## Étapes
1. Chargement des données préprocessées
2. Normalisation des features (StandardScaler)
3. Entraînement avec tracking MLflow
4. Évaluation : AUC, matrice de confusion, coût métier

In [22]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier

## 1. Chargement des données

In [13]:
df = pd.read_csv('../data/train_preprocessed.csv')
print(f'Shape : {df.shape}')
print(f'\nDistribution TARGET :')
print(df['TARGET'].value_counts(normalize=True).round(3))

# Séparation features / target
y = df['TARGET']
X = df.drop(columns=['TARGET'])

# Split stratifié : conserve la proportion de TARGET dans chaque set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nX_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')

Shape : (10000, 227)

Distribution TARGET :
TARGET
0    0.922
1    0.078
Name: proportion, dtype: float64

X_train : (8000, 226)
X_test  : (2000, 226)


## 2. Normalisation (StandardScaler)

La Régression Logistique est sensible à l'échelle des variables.
Sans normalisation, une feature comme `AMT_INCOME_TOTAL` (centaines de milliers)
écraserait un flag 0/1 dans les calculs.

**Règle importante** : on `fit` le scaler **uniquement sur X_train**,
puis on applique la même transformation à X_test.
Fitter sur X_test introduirait du **data leakage** (fuite d'info du futur).

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform sur train
X_test_scaled  = scaler.transform(X_test)        # transform seulement sur test
print('Normalisation OK')

Normalisation OK


## 3. Entraînement — Régression Logistique (baseline)

Choix des paramètres :
- `class_weight='balanced'` : compense le déséquilibre des classes (beaucoup plus de 0 que de 1)
- `max_iter=2000` : assez d'itérations pour que l'algorithme converge
- `solver='lbfgs'` : algorithme d'optimisation adapté aux datasets de taille moyenne

In [ ]:
mlflow.set_experiment('scoring_credit')

params = {
    'solver': 'lbfgs',
    'max_iter': 2000,
    'random_state': 42,
    'class_weight': 'balanced',
}

with mlflow.start_run(run_name='LogisticRegression_baseline'):

    mlflow.set_tag('model', 'LogisticRegression')

    # --- Entraînement ---
    lr = LogisticRegression(**params)
    lr.fit(X_train_scaled, y_train)

    # --- Prédictions ---
    y_pred  = lr.predict(X_test_scaled)
    y_proba = lr.predict_proba(X_test_scaled)[:, 1]  # probabilité d'être classe 1

    # --- Métriques ---
    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp  # FN coûte 10x plus qu'un FP

    # --- Log MLflow ---
    mlflow.log_params(params)
    mlflow.log_metric('auc',         auc)
    mlflow.log_metric('f1',          f1)
    mlflow.log_metric('cout_metier', cout_metier)
    mlflow.log_metric('fn',          fn)
    mlflow.log_metric('fp',          fp)
    mlflow.sklearn.log_model(lr, name='model')

    # --- Affichage ---
    print(f'AUC         : {auc:.4f}  (cible : 0.75 - 0.82)')
    print(f'F1          : {f1:.4f}')
    print(f'Coût métier : {cout_metier}  (FN={fn}, FP={fp})')
    print(f'\n{classification_report(y_test, y_pred)}')

## 4. Entraînement — Random Forest

Le Random Forest est un modèle à base d'**arbres de décision** :
- Il n'est **pas sensible à l'échelle** des variables → pas besoin de StandardScaler
- `n_estimators=100` : 100 arbres construits en parallèle, le vote majoritaire donne la prédiction finale
- `max_depth=10` : limite la profondeur de chaque arbre pour éviter l'overfitting
- `n_jobs=-1` : utilise tous les CPU disponibles pour accélérer l'entraînement

In [ ]:
params_rf = {
    'n_estimators': 100,
    'max_depth': 10,
    'random_state': 42,
    'class_weight': 'balanced',
    'n_jobs': -1,
}

with mlflow.start_run(run_name='RandomForest_baseline'):

    mlflow.set_tag('model', 'RandomForest')

    # --- Entraînement (pas de scaling pour les arbres) ---
    rf = RandomForestClassifier(**params_rf)
    rf.fit(X_train, y_train)

    # --- Prédictions ---
    y_pred  = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]

    # --- Métriques ---
    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp

    # --- Log MLflow ---
    mlflow.log_params(params_rf)
    mlflow.log_metric('auc',         auc)
    mlflow.log_metric('f1',          f1)
    mlflow.log_metric('cout_metier', cout_metier)
    mlflow.log_metric('fn',          fn)
    mlflow.log_metric('fp',          fp)
    mlflow.sklearn.log_model(rf, name='model')

    # --- Affichage ---
    print(f'AUC         : {auc:.4f}  (cible : 0.75 - 0.82)')
    print(f'F1          : {f1:.4f}')
    print(f'Coût métier : {cout_metier}  (FN={fn}, FP={fp})')
    print(f'\n{classification_report(y_test, y_pred)}')

## 5. Entraînement — XGBoost

XGBoost est un algorithme de **gradient boosting** : il construit les arbres **séquentiellement**,
chaque arbre corrigeant les erreurs du précédent (contrairement au RF qui les construit en parallèle).

Choix des paramètres :
- `scale_pos_weight` : remplace `class_weight='balanced'` pour XGBoost. Il vaut `nb_négatifs / nb_positifs` pour compenser le déséquilibre
- `n_estimators=200` : 200 arbres en séquence
- `max_depth=6` : profondeur standard pour le boosting (moins profond que RF car les arbres se corrigent entre eux)
- `learning_rate=0.1` : taille du pas d'apprentissage — plus petit = plus précis mais plus lent
- `use_label_encoder=False` + `eval_metric='logloss'` : suppriment des warnings XGBoost

In [23]:
# Calcul du scale_pos_weight : ratio négatifs / positifs
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight : {scale_pos_weight:.2f}  ({neg} négatifs / {pos} positifs)')

params_xgb = {
    'n_estimators':     200,
    'max_depth':        6,
    'learning_rate':    0.1,
    'scale_pos_weight': scale_pos_weight,
    'random_state':     42,
    'eval_metric':      'logloss',
    'n_jobs':           -1,
}

with mlflow.start_run(run_name='XGBoost_baseline'):

    mlflow.set_tag('model', 'XGBoost')

    # --- Entraînement (pas de scaling pour le boosting) ---
    xgb = XGBClassifier(**params_xgb)
    xgb.fit(X_train, y_train)

    # --- Prédictions ---
    y_pred  = xgb.predict(X_test)
    y_proba = xgb.predict_proba(X_test)[:, 1]

    # --- Métriques ---
    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp

    # --- Log MLflow ---
    mlflow.log_params(params_xgb)
    mlflow.log_metric('auc',         auc)
    mlflow.log_metric('f1',          f1)
    mlflow.log_metric('cout_metier', cout_metier)
    mlflow.log_metric('fn',          fn)
    mlflow.log_metric('fp',          fp)
    mlflow.sklearn.log_model(xgb, name='model')

    # --- Affichage ---
    print(f'AUC         : {auc:.4f}  (cible : 0.75 - 0.82)')
    print(f'F1          : {f1:.4f}')
    print(f'Coût métier : {cout_metier}  (FN={fn}, FP={fp})')
    print(f'\n{classification_report(y_test, y_pred)}')

scale_pos_weight : 11.90  (7380 négatifs / 620 positifs)


2026/03/02 18:22:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC         : 0.7433  (cible : 0.75 - 0.82)
F1          : 0.2589
Coût métier : 1264  (FN=115, FP=114)

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1845
           1       0.26      0.26      0.26       155

    accuracy                           0.89      2000
   macro avg       0.60      0.60      0.60      2000
weighted avg       0.89      0.89      0.89      2000

